In [ ]:
import os
import sys
import time
import pathlib
import textwrap
import base64
import json
import re
from typing import Optional
import subprocess, argparse
from pathlib import Path
from urllib.parse import urlparse

try:
    import paramiko
except ImportError:
    sys.exit("Не найден paramiko. Установите: pip install paramiko")

try:
    import boto3
    from botocore.exceptions import ClientError
except ImportError:
    sys.exit("Не найден boto3. Установите: pip install boto3")

try:
    import pandas as pd
except ImportError:
    sys.exit("Не найден pandas. Установите: pip install pandas")

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass


# ══════════════════════════════════════════════════════════════════════════════
# КОНФИГУРАЦИЯ — заполните или задайте через .env
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    # ── VM / SSH ──────────────────────────────────────────────────────────────
    "VM_HOST":        "81.26.176.250",              # публичный IP VM
    "VM_PORT":        22,
    "VM_USER":        "ubuntu",
    "VM_KEY_PATH":    os.getenv("VM_KEY_PATH", "~/.ssh/id_rsa"), # путь к .pem/.key
    "VM_KEY":         "",

    # ── Пути на VM ────────────────────────────────────────────────────────────
    "VM_PROJECT_DIR": "/home/ubuntu/esg-pipeline",
    "VM_LOG_DIR":     "/home/ubuntu/esg-pipeline/data/logs",

    # ── S3 / Object Storage ───────────────────────────────────────────────────
    "S3_BUCKET":             "esg-pipeline-data",
    "S3_ENDPOINT":           "https://storage.yandexcloud.net",
    "AWS_ACCESS_KEY_ID":     "",
    "AWS_SECRET_ACCESS_KEY": "",
    "AWS_DEFAULT_REGION":    "ru-central1",

    # ── Gateway ───────────────────────────────────────────────────────────────
    "GATEWAY_PORT":   "",
    "CLIENT_API_KEY": "", 
    # ── YandexGPT (краулер F) ─────────────────────────────────────────────────

    'YANDEX_API_KEY': "",
    'YANDEX_FOLDER_ID': ""
}

# Имена sh-скриптов (относительно VM_PROJECT_DIR)
_SCRIPTS = {
    "full":          "run_full_pipeline.sh",
    "parse":         "run_parse.sh",
    "build_sources": "run_build_sources.sh",
    "rag":           "run_rag.sh",
    "download_raw":  "run_download_raw.sh",
    "upload_results": "run_upload_results.sh",
}

S3_RAG_RAW_KEY = "output/rag_by_variable.csv"

# Единый префикс, под которым CSV/PDF/TXT кладутся в S3 и на VM.
EXTERNAL_SOURCES_PREFIX = "esg_urls_"


# ══════════════════════════════════════════════════════════════════════════════
# ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ
# ══════════════════════════════════════════════════════════════════════════════
def _extract_inn(text: str) -> str:
    m = re.search(r'(?<!\d)(\d{10}|\d{12})(?!\d)', text)
    return m.group(0) if m else ""


def _match_normalize(s: str) -> str:
    """Для сравнения id: только буквы/цифры в нижнем регистре."""
    return re.sub(r"[^a-z0-9]", "", s.lower())


def _ensure_inn_prefix(company_id: str) -> str:
    """
    Гарантирует, что company_id начинается с ИНН, БЕЗ дублирования.
    - ИНН уже в начале строки → возвращаем как есть.
    - ИНН найден внутри строки, но не в начале → переносим его в начало.
    - ИНН не найден вообще → возвращаем как есть.
    """
    inn = _extract_inn(company_id)
    if not inn:
        return company_id
    if company_id.startswith(inn):
        return company_id
    return f"{inn}_{company_id}"


def _derive_canonical_id(csv_path) -> str:
    """
    Определяет id компании, сравнивая ДВА источника:
      - filename_id — id, зашитый в имени CSV-файла (esg_urls_<X>.csv → X,
        или просто имя файла без расширения, если префикса нет)
      - content_id  — <INN>_<domain_stem>, вычисленный из содержимого CSV
        (колонка INN + домен из final_url/URL)

    Правило:
      - если нормализованный content_id является подстрокой нормализованного
        filename_id (это покрывает случаи вроде "..._com", "..._ar2024" в
        конце имени файла) — совпадение есть → берём чистый content_id;
      - если не совпадает → берём filename_id как есть (доверяем имени файла).

    Гарантированно возвращает id с ИНН в начале (через _ensure_inn_prefix),
    без дублирования, если ИНН уже был на месте.
    """
    p = pathlib.Path(csv_path)
    stem = p.stem
    filename_id = stem[len(EXTERNAL_SOURCES_PREFIX):] if stem.startswith(EXTERNAL_SOURCES_PREFIX) else stem

    inn, domain_stem = "", ""
    for enc in ("utf-8-sig", "utf-8", "cp1251"):
        for sep in (",", ";"):
            try:
                df = pd.read_csv(p, sep=sep, encoding=enc, dtype=str, nrows=5)
                if df.shape[1] < 3:
                    continue
                if "INN" in df.columns and not inn:
                    for v in df["INN"].dropna():
                        m = re.search(r"(?<!\d)(\d{10}|\d{12})(?!\d)", str(v))
                        if m:
                            inn = m.group(1)
                            break
                url_col = next((c for c in ("final_url", "URL", "url") if c in df.columns), None)
                if url_col and not domain_stem:
                    for v in df[url_col].dropna():
                        host = re.sub(r"^www\.", "", urlparse(str(v)).netloc.lower())
                        if host:
                            domain_stem = host.split(".")[0]
                            break
                if inn and domain_stem:
                    break
            except Exception:
                continue
        if inn and domain_stem:
            break

    if not inn:
        inn = _extract_inn(filename_id)

    content_id = f"{inn}_{domain_stem}" if inn and domain_stem else (inn or domain_stem)

    if content_id and _match_normalize(content_id) in _match_normalize(filename_id):
        print(f"  canonical_id: content_id={content_id!r} совпадает с filename_id={filename_id!r} → беру content_id")
        result = content_id
    else:
        print(f"  canonical_id: content_id={content_id!r} НЕ совпадает с filename_id={filename_id!r} → беру filename_id")
        result = filename_id or content_id or "unknown"

    return _ensure_inn_prefix(result)


def _s3_result_key(company_id: str) -> str:
    return f"output/{_ensure_inn_prefix(company_id)}.csv"


def _ssh_client() -> paramiko.SSHClient:
    """Открыть SSH-соединение к VM и вернуть клиент."""
    host = CONFIG["VM_HOST"]
    if not host:
        sys.exit("VM_HOST не задан. Укажите публичный IP VM в .env или CONFIG.")

    import io
    key_str = CONFIG.get("VM_KEY", "").strip()
    if not key_str:
        sys.exit("VM_KEY не задан. Вставьте содержимое приватного ключа в CONFIG.")

    try:
        pkey = paramiko.Ed25519Key.from_private_key(io.StringIO(key_str))
    except Exception as e:
        sys.exit(f"Не удалось прочитать ключ: {e}")

    client = paramiko.SSHClient()
    client.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    client.connect(
        hostname=host,
        port=CONFIG["VM_PORT"],
        username=CONFIG["VM_USER"],
        pkey=pkey,
        timeout=15,
    )
    return client


def _run_remote(command: str, stream: bool = True, timeout: int = 7200) -> tuple[int, str]:
    """
    Выполнить команду на VM по SSH.
    При stream=True — вывод идёт в терминал построчно в реальном времени.
    Возвращает (exit_code, stdout+stderr).
    """
    client = _ssh_client()
    print(f"\n[SSH] {command}\n{'─' * 60}")
    try:
        transport = client.get_transport()
        transport.set_keepalive(30)

        channel = transport.open_session()
        channel.settimeout(timeout)
        channel.exec_command(command)

        buf = []
        while True:
            if channel.recv_ready():
                chunk = channel.recv(4096).decode("utf-8", errors="replace")
                buf.append(chunk)
                if stream:
                    print(chunk, end="", flush=True)
            if channel.recv_stderr_ready():
                chunk = channel.recv_stderr(4096).decode("utf-8", errors="replace")
                buf.append(chunk)
                if stream:
                    print(chunk, end="", flush=True)
            if channel.exit_status_ready() and not channel.recv_ready() and not channel.recv_stderr_ready():
                break
            time.sleep(0.05)

        code = channel.recv_exit_status()
        return code, "".join(buf)
    finally:
        client.close()


def _s3():
    """Создать boto3-клиент для Yandex Object Storage."""
    return boto3.client(
        "s3",
        endpoint_url=CONFIG["S3_ENDPOINT"],
        aws_access_key_id=CONFIG["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=CONFIG["AWS_SECRET_ACCESS_KEY"],
        region_name=CONFIG["AWS_DEFAULT_REGION"],
    )


def _ok(msg):   print(f"  ✓  {msg}")
def _err(msg):  print(f"  ✗  {msg}", file=sys.stderr)
def _section(t): print(f"\n{'═'*60}\n  {t}\n{'═'*60}")


def _download_parsed_results_to_vm(normalized_files: list) -> bool:
    """
    Скачать только нужные CSV из S3 raw/parsed_results/ на VM — без run_download_raw.sh.
    """
    script = f"""
import os, boto3

files    = {json.dumps(normalized_files)}
bucket   = os.environ["S3_BUCKET"]
endpoint = os.environ.get("S3_ENDPOINT", "https://storage.yandexcloud.net")
dst_dir  = os.environ["PARSED_RESULTS_DIR"]
os.makedirs(dst_dir, exist_ok=True)

s3 = boto3.client("s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "ru-central1"),
)
for name in files:
    key = f"raw/parsed_results/{{name}}"
    out = os.path.join(dst_dir, name)
    s3.download_file(bucket, key, out)
    print("downloaded:", key, "->", out)
"""
    encoded = base64.b64encode(script.encode()).decode()

    cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"set -a && source ~/.env && [ -f .env ] && source .env && set +a && "
        f"python3 -c \"import base64; exec(base64.b64decode('{encoded}').decode())\""
    )

    _section("Sync S3 → VM (точечно)")
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("Файлы синхронизированы")
    else:
        _err(f"Синхронизация завершилась с кодом {code}")
    return code == 0


def _finalize_rag_output_in_s3(company_id: str) -> Optional[str]:
    """
    run_rag.sh / rag_by_variable.py всегда пишут результат под одним и тем же
    именем — output/rag_by_variable.csv. Эта функция переименовывает результат
    в S3 под уникальное для компании имя: output/rag_by_variable.csv →
    output/<company_id>.csv
    """
    s3 = _s3()
    bucket = CONFIG["S3_BUCKET"]
    dst_key = _s3_result_key(company_id)

    try:
        s3.head_object(Bucket=bucket, Key=S3_RAG_RAW_KEY)
    except ClientError as e:
        _err(f"Не найден сырой результат RAG в S3 ({S3_RAG_RAW_KEY}): {e}")
        return None

    try:
        s3.copy_object(
            Bucket=bucket,
            CopySource={"Bucket": bucket, "Key": S3_RAG_RAW_KEY},
            Key=dst_key,
        )
        s3.delete_object(Bucket=bucket, Key=S3_RAG_RAW_KEY)
        _ok(f"Результат переименован в S3: {S3_RAG_RAW_KEY} → {dst_key}")
        return dst_key
    except Exception as e:
        _err(f"Не удалось переименовать результат в S3: {e}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# ПУБЛИЧНЫЕ ФУНКЦИИ
# ══════════════════════════════════════════════════════════════════════════════

def health_check() -> bool:
    """
    Проверить работоспособность всей системы:
      1. SSH-соединение с VM
      2. Сервис llm-gateway (systemctl)
      3. Gateway /health через curl внутри VM
      4. S3-бакет доступен
    """
    _section("Health Check")
    all_ok = True

    print("1. SSH-соединение с VM...")
    try:
        code, out = _run_remote("echo PING && uptime", stream=False, timeout=15)
        if code == 0 and "PING" in out:
            _ok(f"VM доступна  |  {out.strip().splitlines()[-1]}")
        else:
            _err(f"SSH подключился, но команда упала (exit {code})")
            all_ok = False
    except Exception as e:
        _err(f"SSH недоступен: {e}")
        all_ok = False

    print("\n2. Статус сервиса llm-gateway...")
    try:
        code, out = _run_remote(
            "systemctl is-active llm-gateway 2>/dev/null || echo inactive",
            stream=False, timeout=15,
        )
        status = out.strip()
        if status == "active":
            _ok("llm-gateway.service active")
        else:
            _err(f"llm-gateway.service: {status}")
            _err("  Запустить вручную: sudo systemctl start llm-gateway")
            all_ok = False
    except Exception as e:
        _err(f"Не удалось проверить сервис: {e}")
        all_ok = False

    print("\n3. Gateway HTTP /health (curl на VM)...")
    try:
        port = CONFIG["GATEWAY_PORT"]
        api_key = CONFIG["CLIENT_API_KEY"]
        curl_cmd = (
            f'curl -s -o /dev/null -w "%{{http_code}}" '
            f'-H "X-API-Key: {api_key}" '
            f'http://127.0.0.1:{port}/health'
        )
        code, out = _run_remote(curl_cmd, stream=False, timeout=15)
        http_code = out.strip()
        if code == 0 and http_code == "200":
            _ok(f"Gateway /health → HTTP {http_code}")
        else:
            _err(f"Gateway /health → HTTP {http_code or 'нет ответа'} (exit {code})")
            all_ok = False
    except Exception as e:
        _err(f"Не удалось проверить gateway: {e}")
        all_ok = False

    print("\n4. Доступность S3-бакета...")
    try:
        _s3().head_bucket(Bucket=CONFIG["S3_BUCKET"])
        _ok(f"Бакет «{CONFIG['S3_BUCKET']}» доступен")
    except ClientError as e:
        _err(f"S3 ошибка: {e}")
        all_ok = False
    except Exception as e:
        _err(f"S3 недоступен: {e}")
        all_ok = False

    print(f"\n{'─'*60}")
    print("  ✓  Все проверки прошли" if all_ok else "  ✗  Часть проверок не прошла — см. выше")
    return all_ok


def upload_links(local_path: str) -> bool:
    """Загрузить CSV-файл со ссылками в S3: raw/selected_links/<имя_файла>."""
    _section("Upload  →  S3")
    path = pathlib.Path(local_path).expanduser().resolve()
    if not path.exists():
        _err(f"Файл не найден: {path}")
        return False

    s3_key = f"raw/selected_links/{path.name}"
    print(f"  {path}  →  s3://{CONFIG['S3_BUCKET']}/{s3_key}")
    try:
        _s3().upload_file(str(path), CONFIG["S3_BUCKET"], s3_key)
        _ok(f"Загружено: s3://{CONFIG['S3_BUCKET']}/{s3_key}")
        return True
    except Exception as e:
        _err(f"Ошибка загрузки: {e}")
        return False


def upload_parsed_results(local_path: str, canonical_id: str) -> bool:
    """
    Загрузить CSV с результатами локального парсинга в S3:
    raw/parsed_results/esg_urls_<canonical_id>.csv
    """
    _section("Upload parsed results -> S3 (raw/parsed_results)")
    path = pathlib.Path(local_path).expanduser().resolve()
    if not path.exists():
        _err(f"Файл не найден: {path}")
        return False

    s3_key = f"raw/parsed_results/{EXTERNAL_SOURCES_PREFIX}{canonical_id}.csv"
    print(f"  {path}  →  s3://{CONFIG['S3_BUCKET']}/{s3_key}")
    try:
        _s3().upload_file(str(path), CONFIG["S3_BUCKET"], s3_key)
        _ok(f"Загружено: s3://{CONFIG['S3_BUCKET']}/{s3_key}")
        return True
    except Exception as e:
        _err(f"Ошибка загрузки: {e}")
        return False


def run_pipeline(company_id: Optional[str] = None) -> bool:
    """Запустить полный пайплайн на VM (run_full_pipeline.sh)."""
    label = f"  [company: {company_id}]" if company_id else ""
    _section(f"Run Full Pipeline{label}")
    cmd = f"cd {CONFIG['VM_PROJECT_DIR']} && bash {_SCRIPTS['full']}"
    if company_id:
        cmd += f" {company_id}"
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("Пайплайн завершён успешно")
    else:
        _err(f"Пайплайн завершился с кодом {code}")
    return code == 0


def run_parse() -> bool:
    """Запустить только шаг парсинга (run_parse.sh)."""
    _section("Run Parse")
    cmd = f"cd {CONFIG['VM_PROJECT_DIR']} && bash {_SCRIPTS['parse']}"
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("Парсинг завершён успешно")
    else:
        _err(f"Парсинг завершился с кодом {code}")
    return code == 0


def run_build_sources() -> bool:
    """Запустить только шаг build_sources (run_build_sources.sh)."""
    _section("Run Build Sources")
    cmd = f"cd {CONFIG['VM_PROJECT_DIR']} && bash {_SCRIPTS['build_sources']}"
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("Build sources завершён успешно")
    else:
        _err(f"Build sources завершился с кодом {code}")
    return code == 0


def run_rag(company_id: Optional[str] = None) -> bool:
    """Запустить только шаг RAG (run_rag.sh)."""
    label = f"  [company: {company_id}]" if company_id else ""
    _section(f"Run RAG{label}")
    cmd = f"cd {CONFIG['VM_PROJECT_DIR']} && bash {_SCRIPTS['rag']}"
    if company_id:
        cmd += f" {company_id}"
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("RAG завершён успешно")
        if company_id:
            _finalize_rag_output_in_s3(company_id)
    else:
        _err(f"RAG завершился с кодом {code}")
    return code == 0


def download_results(company_id, local_dir: str = ".") -> bool:
    """Скачать результат RAG из S3 на локальный диск."""
    _section("Download Results")
    s3_key = _s3_result_key(company_id)
    dst_dir = pathlib.Path(local_dir).expanduser().resolve()
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst_file = dst_dir / os.path.basename(s3_key)

    print(f"  s3://{CONFIG['S3_BUCKET']}/{s3_key}  →  {dst_file}")
    try:
        _s3().download_file(CONFIG["S3_BUCKET"], s3_key, str(dst_file))
        size_kb = dst_file.stat().st_size / 1024
        _ok(f"Скачано: {dst_file}  ({size_kb:.1f} KB)")
        return True
    except ClientError as e:
        code_str = e.response["Error"]["Code"]
        if code_str == "404":
            _err(f"Файл не найден в S3: {s3_key}")
            _err(f"  Пробуем запасной вариант (ещё не переименованный): {S3_RAG_RAW_KEY}")
            try:
                fallback_dst = dst_dir / os.path.basename(S3_RAG_RAW_KEY)
                _s3().download_file(CONFIG["S3_BUCKET"], S3_RAG_RAW_KEY, str(fallback_dst))
                size_kb = fallback_dst.stat().st_size / 1024
                _ok(f"Скачано (fallback): {fallback_dst}  ({size_kb:.1f} KB)")
                return True
            except Exception as e2:
                _err(f"Fallback тоже не удался: {e2}")
                return False
        else:
            _err(f"S3 ошибка [{code_str}]: {e}")
        return False
    except Exception as e:
        _err(f"Ошибка скачивания: {e}")
        return False

from datetime import date, datetime
from typing import Iterable

def download_files_by_filter(
    prefix: str = "output/",
    date_from: Optional[date] = None,
    date_to: Optional[date] = None,
    name_patterns: Optional[Iterable[str]] = None,
    local_dir: str = ".",
) -> list[str]:
    """
    Скачать файлы из S3 (по умолчанию — из output/), подходящие
    по дате изменения ИЛИ по имени/маске (если указаны оба фильтра —
    файл скачивается, если совпал хотя бы один).

    :param prefix:        префикс в S3, где искать файлы (по умолчанию "output/")
    :param date_from:     нижняя граница LastModified (включительно), напр. date(2025, 1, 1)
    :param date_to:       верхняя граница LastModified (включительно)
    :param name_patterns: список подстрок для поиска в имени файла
                           (регистронезависимо, простое вхождение — не regex)
    :param local_dir:     папка на диске, куда сохранять файлы
    :return: список локальных путей к скачанным файлам
    """
    if not date_from and not date_to and not name_patterns:
        _err("Нужно указать хотя бы один фильтр: date_from/date_to или name_patterns")
        return []

    patterns_lower = [p.lower() for p in name_patterns] if name_patterns else []
    # dst_dir = pathlib.Path(local_dir).expanduser().resolve()
    dst_dir = Path(local_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)

    _section(f"Download by filter: s3://{CONFIG['S3_BUCKET']}/{prefix}")

    s3 = _s3()
    downloaded = []
    try:
        paginator = s3.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=CONFIG["S3_BUCKET"], Prefix=prefix):
            for obj in page.get("Contents", []):
                key = obj["Key"]

                # Пропускаем файлы из подпапок — берём только то, что лежит
                # непосредственно в prefix, без вложенных "/"
                rest = key[len(prefix):] if key.startswith(prefix) else key
                if "/" in rest.strip("/"):
                    continue
                if not rest:  # это сам "каталог"-объект (ключ == prefix)
                    continue

                last_modified = obj["LastModified"].date()
                name = os.path.basename(key)

                date_match = False
                if date_from or date_to:
                    lo_ok = (date_from is None) or (last_modified >= date_from)
                    hi_ok = (date_to is None) or (last_modified <= date_to)
                    date_match = lo_ok and hi_ok

                name_match = any(p in name.lower() for p in patterns_lower)

                if date_match or name_match:
                    dst_file = dst_dir / name
                    try:
                        s3.download_file(CONFIG["S3_BUCKET"], key, str(dst_file))
                        size_kb = dst_file.stat().st_size / 1024
                        _ok(f"Скачано: {key}  →  {dst_file}  ({size_kb:.1f} KB)")
                        downloaded.append(str(dst_file))
                    except Exception as e:
                        _err(f"Ошибка скачивания {key}: {e}")
    except Exception as e:
        _err(f"Ошибка при листинге S3: {e}")
        return downloaded

    if not downloaded:
        print("  (ничего не найдено по заданным фильтрам)")
    else:
        print(f"\n  Итого скачано: {len(downloaded)} файл(ов)")

    return downloaded

def list_s3(prefix: str = "") -> list:
    """Вывести список файлов в S3-бакете по префиксу."""
    _section(f"S3 list:  s3://{CONFIG['S3_BUCKET']}/{prefix}")
    try:
        paginator = _s3().get_paginator("list_objects_v2")
        keys = []
        for page in paginator.paginate(Bucket=CONFIG["S3_BUCKET"], Prefix=prefix):
            for obj in page.get("Contents", []):
                size_kb = obj["Size"] / 1024
                modified = obj["LastModified"].strftime("%Y-%m-%d %H:%M")
                print(f"  {modified}  {size_kb:8.1f} KB  {obj['Key']}")
                keys.append(obj["Key"])
        if not keys:
            print("  (пусто)")
        return keys
    except Exception as e:
        _err(f"Ошибка S3 list: {e}")
        return []


def tail_log(log_name: str = "parser_run.log", lines: int = 50) -> str:
    """Вывести последние строки лог-файла с VM."""
    log_path = f"{CONFIG['VM_LOG_DIR']}/{log_name}"
    _section(f"Log tail: {log_path}  (последние {lines} строк)")
    cmd = f"tail -n {lines} {log_path} 2>&1 || echo 'Файл не найден: {log_path}'"
    _, out = _run_remote(cmd, stream=False)
    print(out)
    return out


def parser(mode="quick", input_file="raw_input_example.csv", num=None):
    """Работа локального парсера."""
    p = Path.cwd()
    sys.path.insert(0, str(p))
    import run_parsing_pipeline

    extra_args = []
    if num:
        extra_args.extend(["--limit-sites", str(num)])
    elif mode == "quick":
        extra_args.extend(["--limit-sites", "1"])

    match mode:
        case "quick":
            sys.argv = [
                "run_parsing_pipeline.py",
                "--companies-csv", str(p / input_file),
                "--crawler", "F",
                "--no-llm",
            ] + extra_args
        case 'full':
            sys.argv = [
                "run_parsing_pipeline.py",
                "--companies-csv", str(p / input_file),
                "--crawler", "F",
            ] + extra_args + (['--crawler-extra', '--yandex_api_key', CONFIG["YANDEX_API_KEY"],
                '--yandex_folder', CONFIG["YANDEX_FOLDER_ID"], '--yandex_model', 'yandexgpt'])
        case "no_crawl":
            sys.argv = [
                "run_parsing_pipeline.py",
                "--companies-csv", str(p / input_file),
                "--skip-crawl"
            ]

    start_ts = time.time() - 1
    code = run_parsing_pipeline.main()

    if code != 0:
        print(f"Парсинг завершился с ошибкой (код {code})")
        return []

    result_files = run_parsing_pipeline.list_result_files(since=start_ts)
    print(f"\nФайлы, созданные/обновлённые в этом запуске ({len(result_files)} шт.):")
    for name in result_files:
        print(f"  {name}")

    return result_files


def _clear_parsed_sources_on_vm() -> bool:
    """
    Очистить локальную папку data/parsed_sources/ на VM перед run_build_sources.sh —
    чтобы run_build_sources.sh не залил в S3 файлы прошлых компаний.
    """
    sources_dir_cmd = (
        f"set -a && source ~/.env && "
        f"[ -f {CONFIG['VM_PROJECT_DIR']}/.env ] && source {CONFIG['VM_PROJECT_DIR']}/.env && "
        f"set +a && echo $PARSED_SOURCES_DIR"
    )
    code, out = _run_remote(sources_dir_cmd, stream=False, timeout=15)
    parsed_sources_dir = out.strip()

    if code != 0 or not parsed_sources_dir:
        _err("Не удалось определить PARSED_SOURCES_DIR на VM")
        return False

    if len(parsed_sources_dir) < 10 or parsed_sources_dir in ("/", "/home", "/home/ubuntu"):
        _err(f"Подозрительный путь PARSED_SOURCES_DIR={parsed_sources_dir!r}, очистку отменяем")
        return False

    _section(f"Clear parsed_sources on VM: {parsed_sources_dir}")

    cmd = f'rm -rf "{parsed_sources_dir:s}"/* "{parsed_sources_dir:s}"/.[!.]* 2>/dev/null || true'
    code, _ = _run_remote(cmd, stream=False, timeout=30)

    if code == 0:
        _ok(f"Очищено: {parsed_sources_dir}")
    else:
        _err(f"Очистка завершилась с кодом {code}")
    return code == 0


def set_gateway_model(scorer=None, auditor=None, validator=None):
    project_env = f"{CONFIG['VM_PROJECT_DIR']}/.env"
    
    _, backup = _run_remote(f"grep 'MODEL_' ~/.env {project_env} 2>/dev/null", stream=False)
    print("Текущие значения (для отката):")
    print(backup)

    sed_cmds = []
    for var, value in (("MODEL_SCORER", scorer), ("MODEL_AUDITOR", auditor), ("MODEL_VALIDATOR", validator)):
        if value:
            for env_file in ("~/.env", project_env):
                sed_cmds.append(f"sed -i 's|^{var}=.*|{var}={value}|' {env_file}")

    sed_cmds.append("sudo systemctl restart llm-gateway")
    sed_cmds.append("sleep 5")

    api_key = CONFIG["CLIENT_API_KEY"]
    sed_cmds.append(
        f"curl -s -o /dev/null -w '%{{http_code}}' "
        f"-H 'X-API-Key: {api_key}' "
        f"http://127.0.0.1:8080/health"
    )

    code, out = _run_remote(" && ".join(sed_cmds), stream=True)
    if code != 0 or "200" not in out:
        _err(f"Gateway не ответил после restart (exit={code}) — проверь вручную")
        return False
    _ok("Gateway перезапущен и отвечает, модели обновлены в обоих .env")
    return True

def upload_txt_sources(local_paths: list[str], company_id: str) -> bool:
    """
    Загрузить TXT-файлы в S3: raw/parsed_results/txt/esg_urls_<company_id>/<имя_файла>.
    """
    _section(f"Upload TXT sources → S3 (raw/parsed_results/txt/{EXTERNAL_SOURCES_PREFIX}{company_id})")

    all_ok = True
    s3 = _s3()
    bucket = CONFIG["S3_BUCKET"]

    for local_path in local_paths:
        path = pathlib.Path(local_path).expanduser().resolve()
        if not path.exists():
            _err(f"Файл не найден: {path}")
            all_ok = False
            continue

        s3_key = f"raw/parsed_results/txt/{EXTERNAL_SOURCES_PREFIX}{company_id}/{path.name}"

        if _s3_key_exists(s3, bucket, s3_key):
            print(f"  пропуск (уже в S3): {s3_key}")
            continue

        print(f"  {path}  →  s3://{CONFIG['S3_BUCKET']}/{s3_key}")

        try:
            _s3().upload_file(str(path), CONFIG["S3_BUCKET"], s3_key)
            _ok(f"Загружено: {s3_key}")
        except Exception as e:
            _err(f"Ошибка загрузки {path.name}: {e}")
            all_ok = False

    return all_ok


def _download_txt_sources_to_vm(company_id: str) -> bool:
    """
    Скачать TXT из S3 raw/parsed_results/txt/esg_urls_<company_id>/ на VM
    в ${PARSED_SOURCES_DIR}/esg_urls_<company_id>/ — ту же папку, куда
    скачиваются и PDF (общая папка "внешних" источников для build_sources.py).
    """
    _section(f"Sync TXT S3 → VM (raw/parsed_results/txt/{EXTERNAL_SOURCES_PREFIX}{company_id})")

    script = f"""
import os, boto3, sys

company_id = {json.dumps(company_id)}
prefix_name = f"{EXTERNAL_SOURCES_PREFIX}{{company_id}}"
bucket   = os.environ["S3_BUCKET"]
endpoint = os.environ.get("S3_ENDPOINT", "https://storage.yandexcloud.net")
dst_dir  = os.path.join(os.environ.get("PARSED_SOURCES_DIR", "/home/ubuntu/esg-pipeline/data/parsed_sources"), prefix_name)
os.makedirs(dst_dir, exist_ok=True)

s3 = boto3.client("s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "ru-central1"),
)

prefix = f"raw/parsed_results/txt/{{prefix_name}}/"
paginator = s3.get_paginator("list_objects_v2")
count = 0
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        name = os.path.basename(key)
        out = os.path.join(dst_dir, name)
        s3.download_file(bucket, key, out)
        print("downloaded:", key, "->", out)
        count += 1

if count == 0:
    print("no txt files found for company:", company_id)
sys.exit(0)
"""
    encoded = base64.b64encode(script.encode()).decode()

    cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"set -a && source ~/.env && [ -f .env ] && source .env && set +a && "
        f"python3 -c \"import base64; exec(base64.b64decode('{encoded}').decode())\""
    )

    code, _ = _run_remote(cmd, stream=True)
    return code == 0

def _s3_key_exists(s3_client, bucket: str, key: str) -> bool:
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        return True
    except ClientError as e:
        if e.response["Error"]["Code"] in ("404", "NoSuchKey", "NotFound"):
            return False
        raise
    except Exception:
        return False

def upload_pdf_sources(local_paths: list[str], company_id: str) -> bool:
    """
    Загрузить PDF-файлы компании в S3: raw/parsed_results/pdfs/esg_urls_<company_id>/<файлы>.
    """
    _section(f"Upload PDF sources → S3 (raw/parsed_results/pdfs/{EXTERNAL_SOURCES_PREFIX}{company_id})")
    all_ok = True
    s3 = _s3()
    bucket = CONFIG["S3_BUCKET"]

    for local_path in local_paths:
        path = pathlib.Path(local_path).expanduser().resolve()
        if not path.exists():
            _err(f"Файл не найден: {path}")
            all_ok = False
            continue

        s3_key = f"raw/parsed_results/pdfs/{EXTERNAL_SOURCES_PREFIX}{company_id}/{path.name}"
       
        if _s3_key_exists(s3, bucket, s3_key):
            print(f"  пропуск (уже в S3): {s3_key}")
            continue
        
        print(f"  {path}  →  s3://{CONFIG['S3_BUCKET']}/{s3_key}")

        try:
            _s3().upload_file(str(path), CONFIG["S3_BUCKET"], s3_key)
            _ok(f"Загружено: {s3_key}")
        except Exception as e:
            _err(f"Ошибка загрузки {path.name}: {e}")
            all_ok = False

    return all_ok


def _download_pdf_sources_to_vm(company_id: str) -> bool:
    """
    Скачать PDF из S3 raw/parsed_results/pdfs/esg_urls_<company_id>/ на VM
    в ${PARSED_SOURCES_DIR}/esg_urls_<company_id>/ — общая папка с TXT.
    """
    prefix_name = f"{EXTERNAL_SOURCES_PREFIX}{company_id}"
    _section(f"Sync PDF S3 → VM (raw/parsed_results/pdfs/{prefix_name})")

    script = f"""
import os, boto3, sys

company_id = {json.dumps(company_id)}
prefix_name = f"{EXTERNAL_SOURCES_PREFIX}{{company_id}}"
bucket   = os.environ["S3_BUCKET"]
endpoint = os.environ.get("S3_ENDPOINT", "https://storage.yandexcloud.net")
dst_dir  = os.path.join(os.environ.get("PARSED_SOURCES_DIR", "/home/ubuntu/esg-pipeline/data/parsed_sources"), prefix_name)
os.makedirs(dst_dir, exist_ok=True)

s3 = boto3.client("s3",
    endpoint_url=endpoint,
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    region_name=os.environ.get("AWS_DEFAULT_REGION", "ru-central1"),
)

prefix = f"raw/parsed_results/pdfs/{{prefix_name}}/"
paginator = s3.get_paginator("list_objects_v2")
count = 0

for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        name = os.path.basename(key)
        out = os.path.join(dst_dir, name)
        s3.download_file(bucket, key, out)
        print("downloaded:", key, "->", out)
        count += 1

if count == 0:
    print("no pdf files found for company:", company_id)
else:
    print("total downloaded:", count)

sys.exit(0)
"""
    encoded = base64.b64encode(script.encode()).decode()

    cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"set -a && source ~/.env && [ -f .env ] && source .env && set +a && "
        f"python3 -c \"import base64; exec(base64.b64decode('{encoded}').decode())\""
    )

    code, _ = _run_remote(cmd, stream=True)
    return code == 0


def _resolve_upload_source(f: str) -> tuple[Path, str]:
    """
    f может быть:
      - просто именем ("company.csv")
      - относительным путём ("Работа/company.csv")
      - абсолютным путём ("/home/.../company.csv")

    Возвращает (путь_к_реальному_файлу_на_диске, "чистое" имя без папок).
    """
    p = Path(f).expanduser()
    name = p.name
    if p.exists():
        return p, name

    fallback = Path.cwd() / "parsed_results_links" / name
    return fallback, name


def run_pipeline_analyse(company_id: Optional[str] = None, files_names_csv=None, files_names_txt=None,
                        files_names_pdf=None, upload=False, download=False,
                        model="gpt://{YANDEX_FOLDER_ID}/qwen3-235b-a22b-fp8/latest") -> bool:
    """
    Запустить пайплайн без парсинга: build_sources + RAG на VM.

    Единый id компании (`canonical_id`) вычисляется из содержимого каждого
    CSV в files_names_csv через _derive_canonical_id() — она сравнивает
    имя файла с данными внутри (INN + домен) и выбирает более надёжный
    вариант. Этот же id используется для имён CSV/PDF/TXT в S3 и на VM,
    а также как company_id для build_sources.py и RAG_COMPANY_FILTER.

    :param company_id: если указан явно — используется как есть (обходит
                        автоматический расчёт); иначе вычисляется из CSV.
    """
    all_ok = True

    if not files_names_csv:
        _err("files_names_csv не задан — нечего обрабатывать")
        return False

    resolved = [_resolve_upload_source(f) for f in files_names_csv]
    print("resolved:", resolved)

    canonical_ids = []
    for local_path, name in resolved:
        if not local_path.exists():
            _err(f"Файл не найден для вычисления canonical_id: {local_path}")
            return False
        cid = _derive_canonical_id(local_path)
        canonical_ids.append(cid)
        print(f"  {name}  ->  canonical_id={cid}")

    if company_id is None:
        company_id = ",".join(dict.fromkeys(canonical_ids))

    # PDF/TXT относятся к первой (основной) компании из files_names_csv
    primary_cid = canonical_ids[0] if canonical_ids else company_id

    _section("Clear VM parsed_sources before build_sources")
    if not _clear_parsed_sources_on_vm():
        _err("Не удалось очистить parsed_sources на VM — прерываем")
        return False

    _section(f"TXT sources [company: {primary_cid}]")
    if files_names_txt:
        if not upload_txt_sources(files_names_txt, primary_cid):
            _err("Не удалось загрузить TXT, прерываем")
            return False

    if not _download_txt_sources_to_vm(primary_cid):
        _err("Не удалось скачать TXT на VM, прерываем")
        return False

    # PDF: та же симметричная логика.
    _section(f"PDF sources [company: {primary_cid}]")
    if files_names_pdf:
        if not upload_pdf_sources(files_names_pdf, primary_cid):
            _err("Не удалось загрузить PDF, продолжаем без них")

    if not _download_pdf_sources_to_vm(primary_cid):
        _err("Не удалось скачать PDF на VM, продолжаем без них")

    if upload:
        _section(f"Upload parsed results → S3  [company: {company_id}]")
        for (local_path, name), cid in zip(resolved, canonical_ids):
            if not local_path.exists():
                _err(f"Файл не найден: {local_path}")
                return False
            if not upload_parsed_results(str(local_path), cid):
                _err(f"Не удалось загрузить {local_path}, прерываем")
                return False

    normalized = [f"{EXTERNAL_SOURCES_PREFIX}{cid}.csv" for cid in canonical_ids]
    parsed_results_only = ",".join(normalized)
    print('Normalized names:', parsed_results_only)
    print('Normalized names2:', normalized)

    if not _download_parsed_results_to_vm(normalized):
        _err("Не удалось синхронизировать raw/parsed_results на VM, прерываем")
        return False

    _section(f"Run Build Sources [company: {company_id}]")

    cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"env PARSED_RESULTS_ONLY=\"{parsed_results_only}\" bash {_SCRIPTS['build_sources']}"
        )
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("Build sources завершён успешно")
    else:
        _err(f"Build sources завершился с кодом {code}")
        all_ok = False

    _section(f"Run RAG [company: {company_id}]")
    set_gateway_model(scorer=model, auditor=model, validator=model)

    cmd = f"cd {CONFIG['VM_PROJECT_DIR']} && bash {_SCRIPTS['rag']} {company_id}"
    code, _ = _run_remote(cmd, stream=True)
    if code == 0:
        _ok("RAG завершён успешно")
        finalized_key = _finalize_rag_output_in_s3(company_id)
        if not finalized_key:
            _err("Не удалось переименовать результат RAG в S3 — "
                 "файл остался как output/rag_by_variable.csv")
            all_ok = False
    else:
        _err(f"RAG завершился с кодом {code}")
        all_ok = False

    if download:
        download_results(company_id=company_id, local_dir=".")
    return all_ok


def run_pipeline_one(file, num=1, download=False):
    """
    Запустить пайплайн на VM для одной компании, пропуская шаг парсинга на VM
    (парсинг выполняется локально через parser(), результаты загружаются в S3).
    """
    _section(f"Run Parser (local)  [company: {file}]")
    files_names = parser(mode="full", input_file=file, num=num)

    if not files_names:
        _err("Парсер не вернул ни одного файла результата — пайплайн остановлен")
        return False

    return run_pipeline_analyse(files_names_csv=files_names, upload=True, download=download)

def run_validator_only(company_id: str, model="gpt://{YANDEX_FOLDER_ID}/qwen3-235b-a22b-fp8/latest") -> bool:
    """
    Точечный запуск валидатора на VM для одной компании — по уже готовым
    промежуточным файлам scorer/auditor (без повторного RAG).

    Сравнение расхождений — по полю Score. Если расхождений нет,
    валидатор не запускается, результат не создаётся и не заливается в S3.
    """
    _section(f"Restore parsed_sources for validator  [company: {company_id}]")
    set_gateway_model(validator=model)

    normalized_file = f"esg_urls_{company_id}.csv"
    if not _download_parsed_results_to_vm([normalized_file]):
        _err("Не удалось скачать parsed_results для восстановления источников, прерываем")
        return False

    if not _clear_parsed_sources_on_vm():
        _err("Не удалось очистить parsed_sources на VM — прерываем")
        return False
    
     # ── ДОБАВЛЕНО: восстановление TXT и PDF из S3 ──────────────────────
    _section(f"Restore TXT/PDF for validator  [company: {company_id}]")
    if not _download_txt_sources_to_vm(company_id):
        _err("Не удалось скачать TXT на VM — продолжаем без них")
    if not _download_pdf_sources_to_vm(company_id):
        _err("Не удалось скачать PDF на VM — продолжаем без них")

    build_cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"env PARSED_RESULTS_ONLY=\"{normalized_file}\" bash {_SCRIPTS['build_sources']}"
    )
    code, _ = _run_remote(build_cmd, stream=True)
    if code != 0:
        _err(f"Build sources (restore) завершился с кодом {code}, прерываем")
        return False
    _ok("parsed_sources восстановлены для валидатора")

    

    _section(f"Run Validator Only  [company: {company_id}]")

    script = f"""
import sys
sys.path.insert(0, "{CONFIG['VM_PROJECT_DIR']}")
import os, boto3
from rag.rag_by_variable import run_validator_only, OUTPUT_CSV

result = run_validator_only({company_id!r})

if result is None:
    print("VALIDATOR_SKIPPED: no Score differences")
else:
    bucket = os.environ["S3_BUCKET"]
    endpoint = os.environ.get("S3_ENDPOINT", "https://storage.yandexcloud.net")
    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
        aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"],
        region_name=os.environ.get("AWS_DEFAULT_REGION", "ru-central1"),
    )
    s3.upload_file(OUTPUT_CSV, bucket, "output/rag_by_variable.csv")
    print(f"VALIDATOR_DONE: {{len(result)}} rows, uploaded to S3")
"""
    encoded = base64.b64encode(script.encode()).decode()
    cmd = (
        f"cd {CONFIG['VM_PROJECT_DIR']} && "
        f"source venv/bin/activate && "  
        f"set -a && source ~/.env && [ -f .env ] && source .env && set +a && "
        f"python3 -c \"import base64; exec(base64.b64decode('{encoded}').decode())\""
    )

    code, out = _run_remote(cmd, stream=True)
    if code != 0:
        _err(f"Validator завершился с кодом {code}")
        return False

    if "VALIDATOR_SKIPPED" in out:
        _ok("Расхождений по Score нет — валидатор пропущен, файл не создавался")
        return True

    if "VALIDATOR_DONE" in out:
        _ok("Валидатор выполнен успешно")
        finalized_key = _finalize_rag_output_in_s3(company_id)
        if not finalized_key:
            _err("Не удалось переименовать результат RAG в S3 — "
                 "файл остался как output/rag_by_variable.csv")
            return False
        return True

    _err("Не удалось определить результат выполнения валидатора")
    return False

In [ ]:
# ТОЧКА ВХОДА

folder_id = CONFIG["YANDEX_FOLDER_ID"]
model=f"gpt://{folder_id}/aliceai-llm"
model_scorer=model
model_auditor=model
model_validator=model

folder_name=Path("/Volumes/Transcend/Работа/2026.09.09 - Парсинг Эстония (10 samples, third batch)")
folder = folder_name/"parsed_results"
print("Основная папка:", folder)
print("Существует:", folder.exists())
print("Существует:", folder_name.exists())

for file_path in folder.glob("*.csv"):
            print(file_path.name)
            if str(file_path.name).startswith("._"):
                continue 
            
            files_txt = []
            files_pdf = []

            print(f"Файл csv: {file_path.name[:-4]}")

            if True:
                folder2 = folder_name/f"TXT/{file_path.name[:-4]}"
                for file_path2 in folder2.glob("*.txt"):
                    if file_path2.name.startswith("._"):
                        continue
                    files_txt.append(file_path2)

            print(files_txt)

            run_pipeline_analyse(files_names_csv=[file_path], files_names_txt=files_txt,
                    files_names_pdf=files_pdf, upload=True, download=False, model=model_scorer)

Основная папка: /Volumes/Transcend/Работа/2026.09.09 - Парсинг Эстония (10 samples, third batch)/parsed_results
Существует: True
Существует: True
10188708_cobalt_legal.csv
Файл csv: 10188708_cobalt_legal
[]
[]
resolved: [(PosixPath('/Volumes/Transcend/Работа/2026.09.09 - Парсинг Эстония (10 samples, third batch)/parsed_results/10188708_cobalt_legal.csv'), '10188708_cobalt_legal.csv')]
  canonical_id: content_id='cobalt' совпадает с filename_id='10188708_cobalt_legal' → беру content_id
  10188708_cobalt_legal.csv  ->  canonical_id=cobalt

════════════════════════════════════════════════════════════
  Clear VM parsed_sources before build_sources
════════════════════════════════════════════════════════════

[SSH] set -a && source ~/.env && [ -f /home/ubuntu/esg-pipeline/.env ] && source /home/ubuntu/esg-pipeline/.env && set +a && echo $PARSED_SOURCES_DIR
────────────────────────────────────────────────────────────

════════════════════════════════════════════════════════════
  Clear parse

In [ ]:
# Скачивание фалов последнего запуска
download_files_by_filter(date_from=date(2026, 9, 16), date_to=date(2026, 9, 16), local_dir="results")


════════════════════════════════════════════════════════════
  Download by filter: s3://esg-pipeline-data/output/
════════════════════════════════════════════════════════════
  ✓  Скачано: output/11044696_evv_ee.csv  →  results/11044696_evv_ee.csv  (20.9 KB)
  ✓  Скачано: output/14114233_juhkentali_hotel_ou.csv  →  results/14114233_juhkentali_hotel_ou.csv  (10.6 KB)
  ✓  Скачано: output/balticagromachinery.csv  →  results/balticagromachinery.csv  (7.6 KB)
  ✓  Скачано: output/cobalt.csv  →  results/cobalt.csv  (20.1 KB)
  ✓  Скачано: output/fifaa.csv  →  results/fifaa.csv  (19.4 KB)
  ✓  Скачано: output/gaasivork.csv  →  results/gaasivork.csv  (15.7 KB)
  ✓  Скачано: output/hvv.csv  →  results/hvv.csv  (15.7 KB)
  ✓  Скачано: output/kadrina.csv  →  results/kadrina.csv  (19.9 KB)
  ✓  Скачано: output/kiviolisoojus.csv  →  results/kiviolisoojus.csv  (14.4 KB)
  ✓  Скачано: output/kroonpress.csv  →  results/kroonpress.csv  (24.3 KB)
  ✓  Скачано: output/lindstromgroup.csv  →  results/lin

['results/11044696_evv_ee.csv',
 'results/14114233_juhkentali_hotel_ou.csv',
 'results/balticagromachinery.csv',
 'results/cobalt.csv',
 'results/fifaa.csv',
 'results/gaasivork.csv',
 'results/hvv.csv',
 'results/kadrina.csv',
 'results/kiviolisoojus.csv',
 'results/kroonpress.csv',
 'results/lindstromgroup.csv',
 'results/narvavesi.csv',
 'results/parnuvesi.csv',
 'results/proekspert.csv',
 'results/saarevesi.csv',
 'results/sakumaja.csv',
 'results/silpower.csv',
 'results/terviseamet.csv',
 'results/tootukassa.csv',
 'results/trimtexstore.csv',
 'results/xn--virtuaalettevte-4sb.csv']